In [7]:
import pandas as pd
import numpy as np
import folium

In [6]:
path_relativo = '../data/processed/'
datos = pd.read_csv(path_relativo + 'andaluces_2_5k.csv')
hospitales = pd.read_csv(path_relativo + 'Hospitales_Completo.csv')
display(hospitales)

,nombre,localidad,provincia,capacidad,latitud,longitud
0,Hospital Virgen del Mar,Almería,Almería,76,36.856070,-2.417615
1,Hospital Universitario Torrecardenas,Almería,Almería,779,36.862595,-2.441142
2,Hospital La Inmaculada,Huércal-Overa,Almería,184,37.401403,-1.942036
3,Hospital Universitario de Poniente,"Ejido, El",Almería,281,36.753477,-2.803836
4,Hospital Mediterráneo,Almería,Almería,89,36.820367,-2.435320
...,...,...,...,...,...,...
128,Hospital Psiquiátrico Penitenciario,Sevilla,Sevilla,184,37.390565,-5.847516
129,Clínica de Salud Mental Miguel de Mañara,Dos Hermanas,Sevilla,18,37.334785,-5.918881
130,Hospital de La Mujer,Sevilla,Sevilla,221,37.363119,-5.978213
131,"Adinfa, Sociedad Cooperativa Andaluza",Coria del Río,Sevilla,25,37.287022,-6.056425


In [8]:
def create_map(datos: pd.DataFrame, hospitales: pd.DataFrame, out_html: str):
    # Asegurar que latitud y longitud son numéricas
    datos['latitud'] = pd.to_numeric(datos['latitud'], errors='coerce')
    datos['longitud'] = pd.to_numeric(datos['longitud'], errors='coerce')

    hospitales['latitud'] = pd.to_numeric(hospitales['latitud'], errors='coerce')
    hospitales['longitud'] = pd.to_numeric(hospitales['longitud'], errors='coerce')

    # Centro del mapa = media de coordenadas de municipios (o conjunta)
    center_lat = datos['latitud'].mean()
    center_lon = datos['longitud'].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=7)

    # --- Municipios en azul ---
    for _, row in datos.iterrows():
        if pd.isna(row['latitud']) or pd.isna(row['longitud']):
            continue
        popup_text = (
            f"<b>{row['municipio']}</b><br>"
            f"Provincia: {row['PROVINCIA']}<br>"
            f"Población: {int(row['poblacion'])}<br>"
            f"Lat, Lon: {row['latitud']:.5f}, {row['longitud']:.5f}"
        )
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=4,
            popup=folium.Popup(popup_text, max_width=250),
            fill=True,
            color='blue',
            fill_color='blue',
            fill_opacity=0.6
        ).add_to(m)

    # --- Hospitales en rojo ---
    for _, row in hospitales.iterrows():
        if pd.isna(row['latitud']) or pd.isna(row['longitud']):
            continue
        popup_text = (
            f"<b>{row['nombre']}</b><br>"
            f"Localidad: {row['localidad']}<br>"
            f"Provincia: {row['provincia']}<br>"
            f"Lat, Lon: {row['latitud']:.5f}, {row['longitud']:.5f}"
        )
        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=6,
            popup=folium.Popup(popup_text, max_width=250),
            fill=True,
            color='red',
            fill_color='red',
            fill_opacity=0.9
        ).add_to(m)

    m.save(out_html)
    return m


In [9]:
m=create_map(datos, hospitales, '../maps/raw/mapa_andalucia_prueba.html')

In [4]:
import pandas as pd
import numpy as np
import folium
from folium.features import DivIcon
import math
import branca.colormap as cm
import os
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager

# --- 1. Carga y preparación de datos ---
path_relativo = '../data/processed/'

try:
    datos = pd.read_csv(path_relativo + 'andaluces_2_5k.csv')
    hospitales = pd.read_csv(path_relativo + 'Hospitales_Completo.csv')
except FileNotFoundError:
    print("Error: No se encuentran los archivos CSV. Usando datos dummy...")
    datos = pd.DataFrame({
        'municipio': ['Sevilla', 'Málaga', 'Pueblo Pequeño Serrano'],
        'PROVINCIA': ['Sevilla', 'Málaga', 'Granada'],
        'poblacion': [688711, 571026, 3000],
        'latitud': [37.3891, 36.7213, 37.0000],
        'longitud': [-5.9845, -4.4214, -3.5000]
    })
    hospitales = pd.DataFrame({
        'nombre': ['Hosp. Virgen del Rocío', 'Hosp. Regional Málaga', 'Hosp. Comarcal'],
        'localidad': ['Sevilla', 'Málaga', 'Motril'],
        'provincia': ['Sevilla', 'Málaga', 'Granada'],
        'latitud': [37.3515, 36.7388, 36.75],
        'longitud': [-5.9642, -4.4195, -3.52],
        'capacidad': [1200, 900, 150]
    })

# --- Verificación de columna de camas ---
columna_camas = 'capacidad' 
if columna_camas not in hospitales.columns:
    print(f"¡ADVERTENCIA! La columna '{columna_camas}' no existe.")
    hospitales[columna_camas] = 100
else:
    hospitales[columna_camas] = hospitales[columna_camas].fillna(hospitales[columna_camas].mean())

# --- 2. Funciones de ayuda para Estilos y Escalado ---

colores_provincia = {
    'Almería': '#e6194b',
    'Cádiz': '#3cb44b',
    'Córdoba': '#ffe119',
    'Granada': '#4363d8',
    'Huelva': '#f58231',
    'Jaén': '#911eb4',
    'Málaga': '#42d4f4',
    'Sevilla': '#f032e6'
}

def get_color_provincia(prov_name):
    prov_key = str(prov_name).strip().capitalize()
    return colores_provincia.get(prov_key, '#808080')

max_poblacion = datos['poblacion'].max()
max_camas = hospitales[columna_camas].max()

def escalar_radio_poblacion(valor, min_radius=3, max_radius=25):
    if pd.isna(valor) or valor <= 0: return min_radius
    normalized = math.sqrt(valor) / math.sqrt(max_poblacion)
    return min_radius + (normalized * (max_radius - min_radius))

def escalar_tamano_cruz_hospital(valor, min_size_px=15, max_size_px=50):
    if pd.isna(valor) or valor <= 0: return min_size_px
    normalized = math.sqrt(valor) / math.sqrt(max_camas)
    return min_size_px + (normalized * (max_size_px - min_size_px))

# --- 3. Función Principal de Creación del Mapa ---

def create_advanced_map(datos: pd.DataFrame, hospitales: pd.DataFrame, out_html: str):
    # Limpieza
    datos['latitud'] = pd.to_numeric(datos['latitud'], errors='coerce')
    datos['longitud'] = pd.to_numeric(datos['longitud'], errors='coerce')
    hospitales['latitud'] = pd.to_numeric(hospitales['latitud'], errors='coerce')
    hospitales['longitud'] = pd.to_numeric(hospitales['longitud'], errors='coerce')

    datos = datos.dropna(subset=['latitud', 'longitud'])
    hospitales = hospitales.dropna(subset=['latitud', 'longitud'])

    center_lat = datos['latitud'].mean()
    center_lon = datos['longitud'].mean()

    # CAMBIO 1: Tiles a 'OpenStreetMap' para tener color
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7, tiles='OpenStreetMap')

    # --- Capa Municipios ---
    municipios_layer = folium.FeatureGroup(name="Municipalities (Population)")

    for _, row in datos.iterrows():
        prov_color = get_color_provincia(row['PROVINCIA'])
        radio_dinamico = escalar_radio_poblacion(row['poblacion'])
        
        # Popup formateado
        popup_text = (
            f"<b>{row['municipio']}</b><br>"
            f"Provincia: {row['PROVINCIA']}<br>"
            f"Population: {int(row['poblacion']):,}"
        )

        folium.CircleMarker(
            location=[row['latitud'], row['longitud']],
            radius=radio_dinamico,
            popup=folium.Popup(popup_text, max_width=250),
            color=prov_color,
            fill=True,
            fill_color=prov_color,
            fill_opacity=0.6,
            weight=1
        ).add_to(municipios_layer)

    municipios_layer.add_to(m)

    # --- Capa Hospitales ---
    hospitales_layer = folium.FeatureGroup(name=f"Candidate Hospitals (Beds)")

    for _, row in hospitales.iterrows():
        camas_val = row[columna_camas]
        icon_size_px = escalar_tamano_cruz_hospital(camas_val)

        popup_text = (
            f"<b>{row['nombre']}</b><br>"
            f"Loc: {row['localidad']} ({row['provincia']})<br>"
            f"Beds (Bj): {int(camas_val)}<br>"
        )

        cruz_html = f"""<div style="
            font-family: Arial, sans-serif;
            font-weight: 900;
            font-size: {icon_size_px}px;
            color: #222222;
            text-shadow: 2px 2px 4px rgba(255,255,255, 0.9), -2px -2px 4px rgba(255,255,255, 0.9);
            text-align: center;
            line-height: {icon_size_px}px;
            width: {icon_size_px}px;
            height: {icon_size_px}px;
            cursor: pointer;
            ">+</div>"""

        folium.Marker(
            location=[row['latitud'], row['longitud']],
            icon=DivIcon(html=cruz_html),
            popup=folium.Popup(popup_text, max_width=250),
            z_index_offset=1000
        ).add_to(hospitales_layer)

    hospitales_layer.add_to(m)
    folium.LayerControl().add_to(m)

    # CAMBIO 2: Leyenda en Inglés y posicionada a la derecha
    # Se cambia 'left: 50px' por 'right: 50px'
    leyenda_html = """
     <div style="position: fixed;
                 bottom: 10px; right: 10px; width: 220px; height: auto;
                 border:2px solid grey; z-index:9999; font-size:14px;
                 background-color:white; opacity: 0.9; padding: 10px; font-family: sans-serif;">
      &nbsp;<b>Legend</b><br>
      &nbsp;<i class="fa fa-circle" style="color:gray"></i>&nbsp;Municipalities (Size ∝ Pop.)<br>
      &nbsp;&nbsp;&nbsp;<small>Color by Province</small><br>
      &nbsp;<b style="font-size:20px; line-height:15px;">+</b>&nbsp;Hospitals (Size ∝ Beds)
     </div>
     """
    m.get_root().html.add_child(folium.Element(leyenda_html))

    m.save(out_html)
    print(f"Mapa interactivo guardado en: {out_html}")
    return m

# --- 4. Función de exportación a imagen ---
def save_html_as_image(html_path, image_path, delay=5):
    print(f"Intentando exportar {html_path} a imagen...")
    abs_html_path = os.path.abspath(html_path)
    try:
        options = webdriver.ChromeOptions()
        options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('window-size=1920x1200')

        driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
        driver.get(f"file:///{abs_html_path}")
        time.sleep(delay)
        driver.save_screenshot(image_path)
        driver.quit()
        print(f"Imagen estática guardada con éxito en: {image_path}")
        return True
    except Exception as e:
        print("-" * 40)
        print(f"AVISO: No se pudo generar la imagen automáticamente: {e}")
        return False

# --- EJECUCIÓN ---
nombre_html = '../maps/raw/mapa_avanzado.html'
nombre_imagen = '../maps/raw/figura_mapa_andalucia.png'
os.makedirs(os.path.dirname(nombre_html), exist_ok=True)

mapa = create_advanced_map(datos, hospitales, nombre_html)
save_html_as_image(nombre_html, nombre_imagen)
display(mapa)

Mapa interactivo guardado en: ../maps/raw/mapa_avanzado.html
Intentando exportar ../maps/raw/mapa_avanzado.html a imagen...
Imagen estática guardada con éxito en: ../maps/raw/figura_mapa_andalucia.png
